# 📊 Visualización desde MySQL — Northwind
### Taller de Programación

---

Combinamos lo visto en las dos clases anteriores:
- Consultamos datos desde MySQL con `query()`
- Visualizamos los resultados con Matplotlib y Seaborn

Usamos la vista `vw_orden_detalle` que ya resuelve todos los JOINs.

## 0. Conexión

In [4]:
!pip install mysql-connector-python


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams['figure.dpi']        = 110
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='tab10')

CONN_PARAMS = {
    'host'    : 'localhost',
    'user'    : 'root',
    'password': '',
    'database': 'northwind',
    'charset' : 'utf8mb4',
}

conn = mysql.connector.connect(**CONN_PARAMS)
print('✅ Conexión abierta')

def query(sql, params=None):
    return pd.read_sql(sql, conn, params=params)

c:\Python\Python312\Lib\site-packages\seaborn\_statistics.py:32: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.2)
  from scipy.stats import gaussian_kde


InterfaceError: 2003: Can't connect to MySQL server on 'localhost:3306' (Errno 10061: No connection could be made because the target machine actively refused it)

---
## 1. Ingresos por país

Primero consultamos, luego visualizamos.

In [ ]:
ing_pais = query("""
    SELECT  CustomerCountry                          AS Pais,
            ROUND(SUM(LineTotal), 2)                 AS Ingresos,
            COUNT(DISTINCT OrderID)                  AS Pedidos
    FROM    vw_orden_detalle
    GROUP BY CustomerCountry
    ORDER BY Ingresos DESC
""")

ing_pais.head()

In [ ]:
plt.figure(figsize=(10, 5))
bars = plt.barh(ing_pais['Pais'], ing_pais['Ingresos'],
                color=sns.color_palette('Blues_d', len(ing_pais)))

for bar in bars:
    w = bar.get_width()
    plt.text(w + 500, bar.get_y() + bar.get_height() / 2,
             f'${w:,.0f}', va='center', fontsize=8)

plt.title('Ingresos totales por país', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Ingresos ($)')
plt.gca().xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
plt.tight_layout()
plt.show()

## 2. Ingresos por categoría de producto

In [ ]:
ing_cat = query("""
    SELECT  CategoryName                             AS Categoria,
            ROUND(SUM(LineTotal), 2)                 AS Ingresos
    FROM    vw_orden_detalle
    GROUP BY CategoryName
    ORDER BY Ingresos DESC
""")

ing_cat

In [ ]:
plt.figure(figsize=(9, 5))
bars = plt.bar(ing_cat['Categoria'], ing_cat['Ingresos'],
               color=sns.color_palette('tab10', len(ing_cat)),
               edgecolor='white')

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 500,
             f'${h:,.0f}', ha='center', fontsize=8)

plt.title('Ingresos por categoría de producto', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Categoría')
plt.ylabel('Ingresos ($)')
plt.xticks(rotation=20, ha='right')
plt.gca().yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
plt.tight_layout()
plt.show()

## 3. Evolución mensual de ingresos

In [ ]:
ing_mes = query("""
    SELECT  DATE_FORMAT(OrderDate, '%Y-%m') AS Mes,
            ROUND(SUM(LineTotal), 2)         AS Ingresos,
            COUNT(DISTINCT OrderID)          AS Pedidos
    FROM    vw_orden_detalle
    GROUP BY Mes
    ORDER BY Mes
""")

ing_mes

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(ing_mes['Mes'], ing_mes['Ingresos'],
         marker='o', linewidth=2.5, color='steelblue', markersize=7)
plt.fill_between(ing_mes['Mes'], ing_mes['Ingresos'],
                 alpha=0.12, color='steelblue')

plt.title('Evolución mensual de ingresos', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Mes')
plt.ylabel('Ingresos ($)')
plt.xticks(rotation=45, ha='right')
plt.gca().yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
plt.tight_layout()
plt.show()

## 4. Ranking de empleados por ingresos

In [ ]:
ing_emp = query("""
    SELECT  CONCAT(FirstName, ' ', LastName)         AS Empleado,
            ROUND(SUM(LineTotal), 2)                 AS Ingresos,
            COUNT(DISTINCT OrderID)                  AS Pedidos
    FROM    vw_orden_detalle
    GROUP BY Empleado
    ORDER BY Ingresos DESC
""")

ing_emp

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Performance del equipo de ventas', fontsize=14, fontweight='bold')

# Ingresos por empleado
colores = sns.color_palette('RdYlGn', len(ing_emp))[::-1]
axes[0].barh(ing_emp['Empleado'][::-1], ing_emp['Ingresos'][::-1],
             color=colores, edgecolor='white')
axes[0].set_title('Ingresos totales', fontweight='bold')
axes[0].set_xlabel('Ingresos ($)')
axes[0].xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Pedidos por empleado
axes[1].barh(ing_emp['Empleado'][::-1], ing_emp['Pedidos'][::-1],
             color=colores, edgecolor='white')
axes[1].set_title('Pedidos gestionados', fontweight='bold')
axes[1].set_xlabel('Nº de pedidos')

plt.tight_layout()
plt.show()

## 5. Distribución del ticket por pedido

In [ ]:
ticket = query("""
    SELECT  OrderID,
            CustomerCountry              AS Pais,
            ROUND(SUM(LineTotal), 2)     AS TicketPedido
    FROM    vw_orden_detalle
    GROUP BY OrderID, CustomerCountry
""")

# Top 6 países por cantidad de pedidos
top_paises = ticket['Pais'].value_counts().head(6).index.tolist()
ticket_top = ticket[ticket['Pais'].isin(top_paises)]

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=ticket_top, x='Pais', y='TicketPedido',
            hue='Pais', legend=False, linewidth=1.5)

plt.title('Distribución del ticket por pedido — top 6 países',
          fontsize=13, fontweight='bold', pad=12)
plt.xlabel('País')
plt.ylabel('Ingresos por pedido ($)')
plt.xticks(rotation=15, ha='right')
plt.gca().yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

---
## Cierre de conexión

In [ ]:
conn.close()
print('🔒 Conexión cerrada')

---
## ✏️ Ejercicios

**Ejercicio 1.** Consultá los ingresos por categoría **y por mes**. Graficá las líneas de evolución mensual, una por categoría.

**Ejercicio 2.** Calculá el **ticket promedio por empleado** (ingresos / pedidos). ¿Coincide el ranking con el de ingresos totales?

**Ejercicio 3.** Graficá un heatmap con los ingresos por **categoría × país** (filas = categoría, columnas = país).

In [ ]:
# Ejercicio 1


In [ ]:
# Ejercicio 2


In [ ]:
# Ejercicio 3
